# Analyse Exploratoire du ISOT Fake News Dataset
## Dataset externe pour test de generalisation hors domaine (OOD)

**Objectif** : Explorer le dataset ISOT (Universite de Victoria, 2018) pour comprendre sa structure, ses distributions et ses caracteristiques textuelles avant de l'utiliser comme jeu de test hors domaine.

**Plan** :
1. Chargement des donnees
2. Analyse des labels de veracite
3. Analyse par type d'article
4. Analyse textuelle des articles
5. Analyse du vocabulaire par classe
6. Comparaison avec LIAR (domain shift)
7. Preprocessing et sauvegarde
8. Synthese

**Source** : Ahmed H., Traore I., Saad S. (2018) — *Detecting opinion spams and fake news using text classification*, Security and Privacy, University of Victoria.

In [24]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from collections import Counter
import re
import string
import warnings
warnings.filterwarnings("ignore")

# Chemins
DATA_DIR = Path("../data/externes/ISOT")
DATA_OUT = Path("../data/traitees")
DATA_OUT.mkdir(parents=True, exist_ok=True)

# Palette de couleurs coherente avec les autres notebooks
COLORS_BIN = ["#e74c3c", "#2ecc71"]
COLORS_BIN_MAP = {"Fake": "#e74c3c", "Real": "#2ecc71"}
COLORS_6 = ["#e74c3c", "#e67e22", "#f39c12", "#3498db", "#2ecc71", "#27ae60"]

print("Imports OK")

Imports OK


## 1. Chargement des donnees

Le dataset ISOT contient **34 108 articles** collectes entre 2015 et 2018 :
- Les **articles reels** proviennent de **Reuters.com** (agence de presse)
- Les **articles faux** proviennent de sources non fiables identifiees par **Politifact**
- **Colonnes** : `type` (categorie numerique), `text` (contenu), `label` (0=Fake, 1=Real), `label_name`

C'est un dataset **binaire** (contrairement a LIAR qui a 6 niveaux de veracite).

In [25]:
df = pd.read_csv(DATA_DIR / "ISOT_fake_news.csv")

print(f"Nombre d'articles : {len(df):,}")
print(f"Colonnes : {list(df.columns)}")
print(f"\n=== Types ===")
print(df.dtypes)
print(f"\n=== Valeurs manquantes ===")
print(df.isnull().sum())
print(f"\n=== Doublons ===")
print(f"Doublons exacts : {df.duplicated().sum()}")
df.head()

Nombre d'articles : 34,108
Colonnes : ['type', 'text', 'label', 'label_name']

=== Types ===
type          float64
text              str
label           int64
label_name        str
dtype: object

=== Valeurs manquantes ===
type          1
text          1
label         0
label_name    0
dtype: int64

=== Doublons ===
Doublons exacts : 4


,type,text,label,label_name
0,1.0,How soon they forget Obama was called out duri...,0,Fake
1,1.0,The Breakfast Club interview started out when ...,0,Fake
2,1.0,After Obama was embarrassed by Donald Trump ac...,0,Fake
3,1.0,Tune in to the Alternate Current Radio Network...,0,Fake
4,1.0,Donald Trump flew in an unmarked jet to a meet...,0,Fake


In [26]:
# Apercu statistique
df.describe(include="all")

,type,text,label,label_name
count,34107.000000,34107,34108.000000,34108
unique,NaN,34103,NaN,2
top,NaN,This Facebook user s comment about Donald Duc...,NaN,Real
freq,NaN,2,NaN,17058
mean,0.499868,NaN,0.500117,NaN
std,0.500007,NaN,0.500007,NaN
min,0.000000,NaN,0.000000,NaN
25%,0.000000,NaN,0.000000,NaN
50%,0.000000,NaN,1.000000,NaN
75%,1.000000,NaN,1.000000,NaN


### Nettoyage initial (NaN et doublons)

Avant toute analyse, on supprime les lignes avec des **valeurs manquantes** dans `text` ou `type`,
ainsi que les **doublons exacts**.

In [27]:
# --- Nettoyage initial ---
n_avant = len(df)

# Supprimer les lignes avec NaN dans text ou type
n_nan = df[["text", "type"]].isna().any(axis=1).sum()
df = df.dropna(subset=["text", "type"])
print(f"Lignes avec NaN supprimees : {n_nan}")

# Supprimer les doublons exacts
n_dup = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Doublons exacts supprimes : {n_dup}")

print(f"Dataset : {n_avant:,} -> {len(df):,} articles")
print(f"Colonnes : {list(df.columns)}")
df.head()

Lignes avec NaN supprimees : 2
Doublons exacts supprimes : 4
Dataset : 34,108 -> 34,102 articles
Colonnes : ['type', 'text', 'label', 'label_name']


,type,text,label,label_name
0,1.0,How soon they forget Obama was called out duri...,0,Fake
1,1.0,The Breakfast Club interview started out when ...,0,Fake
2,1.0,After Obama was embarrassed by Donald Trump ac...,0,Fake
3,1.0,Tune in to the Alternate Current Radio Network...,0,Fake
4,1.0,Donald Trump flew in an unmarked jet to a meet...,0,Fake


## 2. Analyse des labels de veracite

Le dataset ISOT est **binaire** : Fake (0) ou Real (1). Contrairement a LIAR (6 classes remachees en 2), ici le labelling est direct. Verifions l'equilibre des classes.

In [28]:
# Distribution des labels
label_counts = df["label_name"].value_counts()
print("Distribution des labels :")
for lab, n in label_counts.items():
    print(f"  {lab}: {n:>6,}  ({n/len(df)*100:.1f}%)")
print(f"\nRatio Fake/Real : {label_counts.get('Fake',0)/label_counts.get('Real',1):.3f}")

Distribution des labels :
  Real: 17,056  (50.0%)
  Fake: 17,046  (50.0%)

Ratio Fake/Real : 0.999


In [29]:
# Visualisation : Distribution Fake vs Real
fig = px.pie(
    names=label_counts.index,
    values=label_counts.values,
    color=label_counts.index,
    color_discrete_map=COLORS_BIN_MAP,
    title="Distribution Fake / Real — ISOT (34 108 articles)",
    hole=0.4,
    template="plotly_white",
)
fig.update_traces(textinfo="label+percent+value", textfont_size=14)
fig.update_layout(height=420, font=dict(size=13))
fig.show()

## 3. Analyse par type d'article

La colonne `type` indique la **categorie** de l'article. Explorons la repartition par type et l'interaction type x label pour identifier d'eventuels biais de collecte.

In [30]:
# Valeurs uniques de la colonne type
print("=== Valeurs uniques de 'type' ===")
print(df["type"].value_counts())
print(f"\nNombre de types distincts : {df['type'].nunique()}")

# Tableau croise type x label
cross = pd.crosstab(df["type"], df["label_name"], margins=True)
print("\n=== Tableau croise type x label ===")
print(cross)

=== Valeurs uniques de 'type' ===
type
0.0    17056
1.0    17046
Name: count, dtype: int64

Nombre de types distincts : 2

=== Tableau croise type x label ===
label_name   Fake   Real    All
type                           
0.0             0  17056  17056
1.0         17046      0  17046
All         17046  17056  34102


In [31]:
# Visualisation : Nombre d'articles par type et par label
type_label = df.groupby(["type", "label_name"]).size().reset_index(name="count")
type_label["type"] = type_label["type"].astype(str)

fig = px.bar(
    type_label,
    x="type", y="count", color="label_name",
    color_discrete_map=COLORS_BIN_MAP,
    barmode="group",
    title="Nombre d'articles par type et par label — ISOT",
    labels={"type": "Type d'article", "count": "Nombre d'articles", "label_name": "Label"},
    template="plotly_white",
)
fig.update_layout(height=450, font=dict(size=13))
fig.show()

## 4. Analyse textuelle des articles

On analyse la **longueur des textes** (nombre de mots, nombre de caracteres, nombre de phrases) pour chaque classe. Les articles complets d'ISOT sont beaucoup plus longs que les declarations de LIAR (~17 mots en moyenne) — cette difference est un indicateur majeur de **domain shift**.

In [32]:
# Features textuelles de base
df["text"] = df["text"].astype(str)
df["n_chars"] = df["text"].str.len()
df["n_words"] = df["text"].str.split().str.len()
df["n_sentences"] = df["text"].str.count(r'[.!?]+')
df["avg_word_len"] = df["n_chars"] / df["n_words"].clip(lower=1)

print("=== Statistiques textuelles par label ===")
stats = df.groupby("label_name")[["n_words", "n_chars", "n_sentences", "avg_word_len"]].agg(
    ["mean", "median", "std", "min", "max"]
).round(1)
print(stats)

=== Statistiques textuelles par label ===
           n_words                         n_chars                             \
              mean median    std min   max    mean  median     std min    max   
label_name                                                                      
Fake         435.0  381.0  353.5  14  8135  2608.4  2259.0  2193.4  74  51793   
Real         385.5  362.0  277.7   8  5170  2387.1  2244.5  1712.4  49  29768   

           n_sentences                       avg_word_len                   \
                  mean median   std min  max         mean median  std  min   
label_name                                                                   
Fake              23.7   20.0  20.0   0  772          6.0    6.0  0.5  4.5   
Real              21.3   19.0  16.3   0  337          6.2    6.2  0.3  4.8   

                  
             max  
label_name        
Fake        24.0  
Real         9.0  


In [33]:
# Distribution du nombre de mots par article
fig = px.histogram(
    df, x="n_words", color="label_name",
    color_discrete_map=COLORS_BIN_MAP,
    nbins=80, barmode="overlay", opacity=0.65,
    title="Distribution du nombre de mots par article — ISOT",
    labels={"n_words": "Nombre de mots", "label_name": "Label"},
    template="plotly_white",
)
fig.update_layout(height=420, font=dict(size=13))
fig.show()

In [34]:
# Box plot : nombre de mots par label
fig = px.box(
    df, x="label_name", y="n_words", color="label_name",
    color_discrete_map=COLORS_BIN_MAP,
    title="Nombre de mots par article — Fake vs Real (ISOT)",
    labels={"label_name": "Label", "n_words": "Nombre de mots"},
    template="plotly_white",
)
fig.update_layout(height=420, showlegend=False, font=dict(size=13))
fig.show()

In [35]:
# Scatter : nombre de mots vs nombre de phrases
sample = df.sample(5000, random_state=42)
fig = px.scatter(
    sample, x="n_words", y="n_sentences", color="label_name",
    color_discrete_map=COLORS_BIN_MAP,
    opacity=0.4,
    title="Nombre de mots vs phrases — ISOT (echantillon 5K)",
    labels={"n_words": "Nombre de mots", "n_sentences": "Nombre de phrases", "label_name": "Label"},
    template="plotly_white",
)
fig.update_layout(height=450, font=dict(size=13))
fig.show()

## 5. Analyse du vocabulaire par classe

Comparons les **mots les plus frequents** dans les articles faux vs reels pour identifier des marqueurs lexicaux distinctifs. On exclut les stop words anglais courants.

In [36]:
# Stop words (coherent avec EDA_LIAR)
STOP_WORDS = {
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "by", "from", "is", "was", "are", "were", "be", "been",
    "being", "have", "has", "had", "do", "does", "did", "will", "would",
    "could", "should", "may", "might", "shall", "can", "not", "no", "that",
    "this", "it", "he", "she", "they", "we", "you", "i", "his", "her",
    "their", "its", "my", "your", "our", "as", "if", "than", "so", "up",
    "out", "about", "who", "what", "which", "when", "where", "how", "all",
    "each", "more", "also", "after", "before", "said", "s", "t", "new",
    "one", "two", "just", "over", "into", "only", "other", "been", "some",
}

def get_top_words(texts, n=20):
    counter = Counter()
    for text in texts:
        tokens = str(text).lower().translate(str.maketrans("", "", string.punctuation)).split()
        counter.update(w for w in tokens if w not in STOP_WORDS and len(w) > 2)
    return counter.most_common(n)

top_fake = get_top_words(df[df["label"] == 0]["text"], 20)
top_real = get_top_words(df[df["label"] == 1]["text"], 20)

print("=== Top 20 mots — FAKE ===")
for w, c in top_fake:
    print(f"  {w:20s} {c:>8,}")

print("\n=== Top 20 mots — REAL ===")
for w, c in top_real:
    print(f"  {w:20s} {c:>8,}")

=== Top 20 mots — FAKE ===
  trump                  63,243
  people                 20,639
  president              19,068
  donald                 14,735
  there                  14,396
  like                   13,960
  him                    13,786
  clinton                12,636
  obama                  12,609
  because                12,546
  even                   11,030
  them                   10,757
  now                    10,256
  via                    10,037
  white                   9,971
  time                    9,825
  news                    9,821
  hillary                 9,472
  image                   9,167
  state                   8,838

=== Top 20 mots — REAL ===
  trump                  39,482
  president              21,641
  house                  15,399
  republican             15,038
  state                  14,821
  states                 13,423
  government             12,719
  united                 12,268
  trump’s                11,506
  told           

In [37]:
# Visualisation comparative
fig = make_subplots(rows=1, cols=2, subplot_titles=("Top 20 mots — Fake", "Top 20 mots — Real"))

for i, (top, color, name) in enumerate([
    (top_fake, "#e74c3c", "Fake"),
    (top_real, "#2ecc71", "Real"),
]):
    words, counts = zip(*reversed(top))
    fig.add_trace(
        go.Bar(y=list(words), x=list(counts), orientation="h",
               marker_color=color, name=name),
        row=1, col=i+1
    )

fig.update_layout(
    title="Vocabulaire le plus frequent par classe — ISOT",
    height=550, template="plotly_white", showlegend=False, font=dict(size=12)
)
fig.show()

## 6. Comparaison avec LIAR (domain shift)

Le dataset LIAR contient des **declarations courtes** (~17 mots) de politiciens, tandis qu'ISOT contient des **articles complets** (~400+ mots). Cette difference structurelle constitue le **domain shift** que nous cherchons a mesurer lors de l'evaluation OOD.

In [38]:
# Tableau comparatif LIAR vs ISOT
liar_path = Path("../data/traitees/liar_complet.parquet")
if liar_path.exists():
    df_liar = pd.read_parquet(liar_path)
    df_liar["n_words"] = df_liar["statement"].astype(str).str.split().str.len()

    comparison = pd.DataFrame({
        "Metrique": [
            "Nombre d'exemples", "Mots (mediane)", "Mots (moyenne)",
            "Mots (max)", "Nb classes", "Type de contenu", "Source", "Langue",
        ],
        "LIAR": [
            f"{len(df_liar):,}", f"{df_liar['n_words'].median():.0f}",
            f"{df_liar['n_words'].mean():.0f}", f"{df_liar['n_words'].max():,}",
            "6 (remache en 2)", "Declarations courtes", "PolitiFact", "Anglais",
        ],
        "ISOT": [
            f"{len(df):,}", f"{df['n_words'].median():.0f}",
            f"{df['n_words'].mean():.0f}", f"{df['n_words'].max():,}",
            "2 (Fake / Real)", "Articles complets", "Reuters + non fiable", "Anglais",
        ],
    })
    print("=== Comparaison LIAR vs ISOT ===")
    print(comparison.to_string(index=False))
else:
    print("Fichier LIAR non trouve — executez EDA_LIAR d'abord")

=== Comparaison LIAR vs ISOT ===
         Metrique                 LIAR                 ISOT
Nombre d'exemples               12,791               34,102
   Mots (mediane)                   17                  373
   Mots (moyenne)                   18                  410
       Mots (max)                  467                8,135
       Nb classes     6 (remache en 2)      2 (Fake / Real)
  Type de contenu Declarations courtes    Articles complets
           Source           PolitiFact Reuters + non fiable
           Langue              Anglais              Anglais


## 7. Preprocessing et sauvegarde

On applique un nettoyage minimal puis on sauvegarde en format **Parquet** pour reutilisation dans les notebooks d'evaluation hors domaine.

In [39]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text_clean"] = df["text"].apply(clean_text)

# Suppression doublons et textes vides
n_before = len(df)
df = df[df["text_clean"].str.len() > 10].drop_duplicates(subset=["text_clean"])
print(f"Nettoyage : {n_before:,} -> {len(df):,} articles ({n_before - len(df)} supprimes)")

# Sauvegarde
out_path = DATA_OUT / "isot_clean.parquet"
cols_keep = ["text_clean", "label", "label_name", "type", "n_words", "n_chars", "n_sentences"]
df[cols_keep].to_parquet(out_path, index=False)
print(f"Sauvegarde -> {out_path}")
print(f"Shape finale : {df[cols_keep].shape}")

Nettoyage : 34,102 -> 34,099 articles (3 supprimes)
Sauvegarde -> ..\data\traitees\isot_clean.parquet
Shape finale : (34099, 7)


## 8. Synthese de l'EDA

**Observations cles** :

1. **Taille** : 34 108 articles — ~2.7x plus grand que LIAR (12 791)
2. **Equilibre** : Quasi-equilibre (~50/50 Fake/Real), pas besoin de SMOTE
3. **Longueur** : Articles complets (~400-600 mots en moyenne vs ~17 pour LIAR) — domain shift majeur
4. **Vocabulaire** : Les articles faux et reels ont des marqueurs lexicaux distincts
5. **Structure** : Pas de metadonnees (speaker, parti) contrairement a LIAR
6. **Domain shift** : Tres fort par rapport a LIAR — degradation attendue en OOD

**Impact pour la modelisation** :
- Le modele entraine sur LIAR (declarations courtes) sera teste sur des articles longs
- La difference de longueur seule peut fausser les predictions des modeles TF-IDF
- Interet pedagogique : mesurer a quel point un modele generalise au-dela de son domaine d'entrainement